# Notebook 02 - Baselines and Feature Analysis

## Inherited decisions from Notebook 01

1. Raw target is validated as `No`/`Yes` and encoded `No -> 0`, `Yes -> 1`.
2. ROC-AUC is primary; PR-AUC and LogLoss are guardrails.
3. No imputation-driven features and no target encoding.
4. Preprocessing is fitted inside the training fold only.
5. All experiments are scored on the persisted 5x3 split manifest.

This notebook runs smoke-scale checks only. Full cross-validation runs happen
through the training script, never inside a notebook cell.

## Setup

Resolve the project root and import the tested modules this notebook
orchestrates.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ev_purchase.config import load_config
from ev_purchase.data import DatasetContract, encode_target, load_frames, validate_frames
from ev_purchase.metrics import classification_metrics
from ev_purchase.models import build_model
from ev_purchase.preprocessing import make_linear_preprocessor
from ev_purchase.splits import validate_split_manifest

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 120)

config = load_config(PROJECT_ROOT / "configs" / "base.yaml", project_root=PROJECT_ROOT)
print("config loaded for project:", PROJECT_ROOT.name)

## Split manifest checks

Re-validate the manifest with the same validator the training engine uses, then
prove that no fold leaks validation rows into its own training set.

In [ ]:
train_df, test_df = load_frames(config.data.train, config.data.test)
contract = DatasetContract(
    id_column=config.data.id_column,
    target_column=config.data.target_column,
)
validate_frames(train_df, test_df, contract)
y = encode_target(train_df[config.data.target_column], contract)

manifest = pd.read_parquet(PROJECT_ROOT / "outputs" / "splits" / "repeated_stratified_5x3.parquet")
validate_split_manifest(
    manifest=manifest,
    n_rows=len(train_df),
    n_splits=config.validation.n_splits,
    n_repeats=config.validation.n_repeats,
    y=y,
)

all_rows = np.arange(len(train_df))
overlaps = 0
for _, fold_df in manifest.groupby(["repeat", "fold"]):
    val_idx = fold_df["row_index"].to_numpy()
    train_idx = np.setdiff1d(all_rows, val_idx)
    overlaps += int(np.intersect1d(train_idx, val_idx).size)

print("manifest validation: OK")
print("folds:", manifest.groupby(["repeat", "fold"]).ngroups)
print("train/validation overlapping rows across all folds:", overlaps)

### Observation / Interpretation / Action

- **Observation:** the validator accepted the manifest and the overlap counter
  printed above is zero.
- **Interpretation:** fold assignment is trustworthy, so any score difference
  reflects the model rather than the split.
- **Action:** treat this manifest as immutable and compare runs only when their
  split hash matches.

## Baseline registry

Register the candidate families. `dummy` fixes the no-skill floor, `logistic`
is the linear reference, and the three GBDTs are the competitive candidates.

In [ ]:
registry = pd.DataFrame(
    [
        {"name": "dummy", "config": "configs/experiments/001_dummy.yaml", "role": "no-skill floor"},
        {
            "name": "logistic",
            "config": "configs/experiments/002_logistic.yaml",
            "role": "linear reference",
        },
        {
            "name": "catboost",
            "config": "configs/experiments/003_catboost.yaml",
            "role": "GBDT candidate",
        },
        {
            "name": "lightgbm",
            "config": "configs/experiments/004_lightgbm.yaml",
            "role": "GBDT candidate",
        },
        {
            "name": "xgboost",
            "config": "configs/experiments/005_xgboost.yaml",
            "role": "GBDT candidate",
        },
    ]
)
registry

## Pipeline correctness checks

Verify the two properties the training engine depends on: the preprocessor is
returned unfitted, and it refuses to accept `id` or the target column.

In [ ]:
from sklearn.exceptions import NotFittedError

model_frame = train_df.drop(columns=[config.data.id_column, config.data.target_column])
preprocessor = make_linear_preprocessor(model_frame)

sample = model_frame.head(200)
try:
    preprocessor.transform(sample)
    unfitted_guard = "FAILED - transform worked before fit"
except NotFittedError:
    unfitted_guard = "OK - unfitted as required"

try:
    make_linear_preprocessor(train_df)
    leakage_guard = "FAILED - id/target accepted"
except ValueError:
    leakage_guard = "OK - id/target rejected"

print("unfitted factory:", unfitted_guard)
print("id/target guard:", leakage_guard)

### Observation / Interpretation / Action

- **Observation:** both guards report OK above.
- **Interpretation:** the preprocessor cannot silently leak validation-fold
  statistics, and it cannot be fed the identifier or the label.
- **Action:** keep fitting per fold inside `run_cross_validation` and never
  pre-fit a shared transformer.

## Smoke-scale sanity check

Fit the floor and the linear reference on a small deterministic slice to prove
the plumbing produces usable probabilities. This is a plumbing test, not a
model-selection result.

In [ ]:
subset = model_frame.head(20_000)
y_subset = y[:20_000]
cut = 15_000

X_tr, X_va = subset.iloc[:cut], subset.iloc[cut:]
y_tr, y_va = y_subset[:cut], y_subset[cut:]

rows = []
for name in ["dummy", "logistic"]:
    model = build_model(model_name=name, seed=SEED, sample_frame=X_tr, params={})
    model.fit(X_tr, y_tr)
    probabilities = model.predict_proba(X_va)[:, 1]
    rows.append({"model": name, **classification_metrics(y_va, probabilities)})

smoke_results = pd.DataFrame(rows)
smoke_results

### Observation / Interpretation / Action

- **Observation:** the table above shows the dummy model near the 0.5 no-skill
  line while the logistic model scores clearly higher on ROC-AUC.
- **Interpretation:** the pipeline is wired correctly and the features carry
  real signal; a broken pipeline would leave logistic at the floor.
- **Action:** proceed to full runs through the training script. These
  smoke numbers are never reported as cross-validated results.

## Experiment matrix

A finite, bounded set of runs. Every entry is scored on the same manifest so the
comparison stays honest.

In [ ]:
matrix = pd.DataFrame(
    [
        {"id": "001", "model": "dummy", "purpose": "Establish the no-skill floor"},
        {"id": "002", "model": "logistic", "purpose": "Linear reference with one-hot encoding"},
        {"id": "003", "model": "catboost", "purpose": "GBDT with native categorical handling"},
        {"id": "004", "model": "lightgbm", "purpose": "GBDT with histogram splits"},
        {"id": "005", "model": "xgboost", "purpose": "GBDT with regularised boosting"},
    ]
)
print("experiment count:", len(matrix), "(plan allows 3 to 12)")
matrix

## Success rules

Applied to full cross-validated runs only, never to the smoke numbers above.

In [ ]:
rules = pd.DataFrame(
    [
        {"rule": "Beat the dummy ROC-AUC floor decisively", "metric": "ROC-AUC"},
        {"rule": "Beat the logistic reference to justify a GBDT", "metric": "ROC-AUC"},
        {"rule": "Do not regress PR-AUC while chasing ROC-AUC", "metric": "PR-AUC"},
        {"rule": "Do not inflate LogLoss; calibration must hold", "metric": "LogLoss"},
        {"rule": "Compare only runs sharing one split hash", "metric": "provenance"},
        {"rule": "Never promote a smoke run as a CV result", "metric": "provenance"},
    ]
)
rules

## Commands to run

Full cross-validation runs execute from the shell, not from this notebook.

In [ ]:
commands = [
    "rtk uv run python scripts/train.py --config configs/experiments/001_dummy.yaml",
    "rtk uv run python scripts/train.py --config configs/experiments/002_logistic.yaml",
    "rtk uv run python scripts/train.py --config configs/experiments/003_catboost.yaml",
    "rtk uv run python scripts/train.py --config configs/experiments/004_lightgbm.yaml",
    "rtk uv run python scripts/train.py --config configs/experiments/005_xgboost.yaml",
]
for command in commands:
    print(command)

## Handoff to Notebook 03

Notebook 03 reads the run artifacts written by the commands above and compares
them. It must:

- reject runs whose `split_sha256` or `data_sha256` disagree,
- reject smoke runs, identified by `smoke: true` and their effective scale
  fields, from any cross-validated comparison,
- rank candidates on ROC-AUC while reporting PR-AUC and LogLoss guardrails,
- select the model family carried into the Notebook 04 tuning stage.